# Text Translation (NLLB)

Translates between languages supported by NLLB. The model is relatively large, so first load can take time and memory. Language codes are case-sensitive.

In [ ]:
%pip install -q transformers sentencepiece torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"
translator.to(device)
print("Loaded on", device)

In [ ]:
LANGUAGES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Marathi": "mar_Deva",
    "French": "fra_Latn", "German": "deu_Latn", "Spanish": "spa_Latn",
    "Gujarati": "guj_Gujr", "Bengali": "ben_Beng", "Tamil": "tam_Taml",
    "Telugu": "tel_Telu", "Kannada": "kan_Knda", "Punjabi": "pan_Guru",
    "Urdu": "urd_Arab"
}
print("Available languages:", ", ".join(LANGUAGES.keys()))
text = input("Text to translate: ").strip()
source_name = input("Source language (e.g. English): ").strip().title()
target_name = input("Target language (e.g. Hindi): ").strip().title()
if source_name not in LANGUAGES or target_name not in LANGUAGES:
    raise ValueError("Choose language names from the displayed list.")
tokenizer.src_lang = LANGUAGES[source_name]
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
with torch.no_grad():
    generated = translator.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(LANGUAGES[target_name]), max_new_tokens=256)
print("Translated:", tokenizer.batch_decode(generated, skip_special_tokens=True)[0])